# 5. PPT — Reporte Estatal

Misma mecánica que `6_PPT_Reporte_Diario.ipynb`: el cuaderno **declara** lo que necesita y
arma la presentación a partir de `input/plantillas/reporte_estados.pptx`. No calcula
indicadores por su cuenta; consume el universo cacheado que produce `src/pipeline`.

Las dos diferencias con el reporte diario:

1. **El perfil es `estatal`**, que comparte por completo la capa A con `reporte_diario`:
   si el notebook 6 (o el 4) ya corrió, aquí hay acierto de caché y el matching no se
   vuelve a ejecutar.
2. **Se genera una PPTX por entidad.** Las rebanadas se declaran una sola vez, en
   versión nacional, y se acotan al vuelo con `Slice.por_entidad(cve)`. Las 32 entidades
   son máscaras booleanas sobre el mismo universo, no 32 corridas de matching.

Los marcadores de la plantilla NO llevan sufijo `_entNN` —son los mismos nombres
nacionales del catálogo—, así que al armar cada PPTX se calcula con la rebanada de la
entidad y se publica con el nombre sin sufijo. Es lo que permite que una sola plantilla
sirva para las 33 presentaciones.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append('../1_funciones')

import re
import unicodedata
from datetime import datetime
from pathlib import Path

from IPython.display import Image, display

import pandas as pd
import numpy as np

# `Presentation` y compañía se usan para dos cosas que el motor de PPT no cubre:
# auditar qué marcadores trae la plantilla y colocar la gráfica de la lámina 4
# en su recuadro (que viene sin marcador).
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from pptx.util import Emu

from src.settings import Settings
from src.pipeline import Universo
from src.slices import Slice
from src.ppt_engine import PPTReportBuilder, dimensiones_placeholder
from src.utils import formatear_fecha_es

from barras_verticales import barras_verticales

## 1. Configuración de esta PPT

El perfil es `perfiles/estatal.toml`. Su `estado = "nacional"` no es una contradicción:
el universo SIEMPRE se construye nacional y la entidad entra después, como dimensión de
rebanada. Cambiar aquí de perfil cambia la configuración de todo el pipeline sin tocar
`config.py` ni el notebook 4.

In [ ]:
import config

PERFIL = "estatal"

# La plantilla se declara aquí, no en el armado: la sección 6 la necesita antes,
# para generar la gráfica con las medidas de su recuadro.
PLANTILLA  = config.PLANTILLA_REPORTE_ESTATAL

# La plantilla es un insumo: si falta, no hay nada que armar.
if not PLANTILLA.exists():
    raise FileNotFoundError(
        f"\u274c No se encontró la plantilla del reporte estatal.\n"
        f"   Ruta declarada en config.PLANTILLA_REPORTE_ESTATAL: {PLANTILLA}"
    )
SALIDA_DIR = config.PPT_ESTATAL_DIR

# Qué presentaciones se generan. `None` es la nacional: misma plantilla, sin filtro de
# entidad, y sirve de control (sus cifras deben coincidir con las del reporte diario).
# La clave 33 ("Nacido en el Extranjero") existe en el RNPDNO y se puede añadir a la
# lista; no se incluye por defecto porque no es una entidad federativa.
ENTIDADES_A_GENERAR = [None] + [cve for cve in sorted(config.ENTIDADES) if cve <= 32]

cfg = Settings.desde_perfil(PERFIL)
print(cfg.resumen())

## 2. Rebanadas requeridas

**Sólo se calcula lo que está en esta lista.** Se declaran en versión NACIONAL y la
sección 4 las acota a cada entidad; agregar una rebanada cuesta microsegundos y ninguna
obliga a recalcular el matching.

Dimensiones disponibles:

- `base`: `rn`, `ba`, `rn-no-ba`, `rn-si-ba`, `ba-no-rn`, `ba-si-rn`
- `periodo`: llaves de `PERIODOS_TIEMPO` (`01ene06-max`, `min-max`, …)
- `estatus`: `des`, `loc`, `all`
- `subfiltro`: banderas `is_*` combinables con `_` (`cisf_sind_fis_cfval` exige las cuatro)
- `estado`: clave INEGI o `None` para nacional

In [ ]:
# Marcadores {{...}} de input/plantillas/reporte_estados.pptx.
# Se calcula SÓLO lo declarado aquí; agregar una rebanada cuesta microsegundos.
# Toda la plantilla cuelga de dos períodos: el histórico completo (`min-max`, lámina 2)
# y el corte de 2006 (`01ene06-max`, láminas 2 a 4).

S_DES_06 = Slice("rn", "01ene06-max", estatus="des")   # reg_rn_01ene06-max_des
S_SISF   = S_DES_06.con(subfiltro="sisf")              # sin información suficiente
S_CISF   = S_DES_06.con(subfiltro="cisf")              # su complemento
S_CIND   = S_DES_06.con(subfiltro="cisf_cind")         # con indicios (registro en PUI)
S_SIND   = S_DES_06.con(subfiltro="cisf_sind")         # sin indicios: el eje de las láminas 3 y 4
S_FIS    = S_SIND.con(subfiltro="cisf_sind_fis")       # …reportados por fiscalías
S_COM    = S_SIND.con(subfiltro="cisf_sind_com")       # …por comisiones de búsqueda
S_POR    = S_SIND.con(subfiltro="cisf_sind_por")       # …por el portal (el residuo)
# Localizados con expediente de fiscalía: el recuadro "Localizados de fiscalías"
# de la lámina 4 (`01ene06-max`), su histórico entre paréntesis (`min-max`) y la
# tercera serie de la gráfica. Ojo: es `loc_fis`, no `loc_cisf_sind_fis` —la
# población del rótulo, no la de las láminas 3 y 4.
S_LOC_FIS_06 = Slice("rn", "01ene06-max", estatus="loc", subfiltro="fis")
S_LOC_FIS_MM = Slice("rn", "min-max",     estatus="loc", subfiltro="fis")

CONTEOS = [
    # ── Lámina 2 · universo histórico completo ───────────────────────────
    Slice("rn", "min-max"),
    Slice("rn", "min-max", estatus="des"),
    Slice("rn", "min-max", estatus="loc"),
    # Localizados: por delito y por estado de vida.
    Slice("rn", "min-max", estatus="loc", subfiltro="cdel"),
    Slice("rn", "min-max", estatus="loc", subfiltro="sdel"),
    Slice("rn", "min-max", estatus="loc", subfiltro="cvid"),
    Slice("rn", "min-max", estatus="loc", subfiltro="svid"),
    # ── Lámina 2 · corte de 2006 ────────────────────────────────────────
    # La plantilla rotula estos dos como "Casos de larga data del año 1952 a
    # 2005" y "Desaparecidos 2006 - 2026", así que el corte es 2006.
    Slice("rn", "min-01ene06", estatus="des"),
    S_DES_06,
    # Información suficiente: `sisf` (ni CURP válida ni registro en PUI) y su
    # complemento `cisf`, que cind/sind parte en dos ramas que suman cisf:
    #   cisf_cind + cisf_sind = cisf   y   cisf + sisf = ..._des
    S_SISF,
    S_SISF.con(subfiltro="sisf_scurp"),   # dentro de `sisf` es el 100 %: `sisf` ya lo exige
    S_CISF,
    S_CIND,
    S_SIND,
    # ── Láminas 3 y 4 · origen del reporte ──────────────────────────────
    # `por` es el residuo: ni fiscalía ni comisión, así que fis + com + por = cisf_sind.
    S_FIS,
    S_FIS.con(subfiltro="cisf_sind_fis_cfval"),
    S_FIS.con(subfiltro="cisf_sind_fis_sfval"),
    S_COM,
    S_POR,
    # ── Lámina 4 · localizados de fiscalías ─────────────────────────────
    S_LOC_FIS_06,
    S_LOC_FIS_MM,
]

# Porcentajes: cada uno se calcula contra su denominador `Slice.padre` —la misma
# rebanada sin su último filtro—, salvo donde se pasa uno explícito. Emiten la
# clave `pct_<nombre>`.
PORCENTAJES = [
    # ── Regla del padre: contra el nivel inmediato superior ──────────────
    Slice("rn", "min-max", estatus="des"),                        # / reg_rn_min-max
    Slice("rn", "min-max", estatus="loc"),                        # / reg_rn_min-max
    Slice("rn", "min-max", estatus="loc", subfiltro="cdel"),      # / ..._loc
    Slice("rn", "min-max", estatus="loc", subfiltro="sdel"),      # / ..._loc
    Slice("rn", "min-max", estatus="loc", subfiltro="cvid"),      # / ..._loc
    Slice("rn", "min-max", estatus="loc", subfiltro="svid"),      # / ..._loc
    S_SISF,                                                       # / reg_rn_01ene06-max_des
    S_CISF,                                                       # / reg_rn_01ene06-max_des
    S_FIS,                                                        # / ..._cisf_sind
    S_FIS.con(subfiltro="cisf_sind_fis_cfval"),                   # / ..._cisf_sind_fis
    S_FIS.con(subfiltro="cisf_sind_fis_sfval"),                   # / ..._cisf_sind_fis
    S_COM,                                                        # / ..._cisf_sind
    S_POR,                                                        # / ..._cisf_sind

    # ── Denominador explícito · las tres ramas de información suficiente ─
    # La regla del padre mediría cisf_cind y cisf_sind contra `..._des_cisf`.
    # La lámina las presenta junto a `sisf` como reparto del total 2006-max,
    # así que las tres se miden contra los desaparecidos de ese período.
    (S_CIND, S_DES_06),
    (S_SIND, S_DES_06),
]

# Los marcadores `_v2` son las MISMAS dos ramas medidas contra `cisf` en vez de
# contra el total: así cind + sind = 100 %, que es como las presenta la lámina 2.
# Van aparte porque `Slice.nombre` no sabe emitir el sufijo `_v2`; la sección 4
# los calcula y los publica con esa clave.
PORCENTAJES_V2 = {S_CIND: S_CISF, S_SIND: S_CISF}

# ── Datos faltantes (lámina 2) ───────────────────────────────────────────────
# El recuadro "Datos faltantes" desglosa, dentro de `sisf`, cuántos registros no
# traen cada campo de identificación. No son banderas `is_*` del pipeline: se
# cuentan aquí sobre las columnas del universo, y por eso viajan por `extra=`.
#
# La regla es contar sobre la columna CRUDA del RNPDNO, no sobre la derivada: lo
# que se quiere saber es cuántos registros llegaron sin el dato, y las derivadas
# del ETL rellenan o concatenan y tapan justamente eso.
#
# `sfdes` daba 0 porque se contaba sobre `fecha_dt`, que sale de
# `fecha_hechos_modificada`: el ETL del notebook 1 la rellena con la fecha de
# percato o de captura y la deja sin un solo nulo. `fechahechos_derivada_rnpdno` es
# la fecha de hechos del RNPDNO sin rellenar y sí conserva los huecos.
#
# `ssexo` cuenta vacíos, NO los 'INDETERMINADO', que son un valor declarado y no
# un dato ausente (para contarlos, añade el `.eq("INDETERMINADO")`).
COLS_FDES_CRUDA = ["fechahechos_derivada_rnpdno"]

# `snomb` se calcula igual: sobre los tres campos crudos del RNPDNO en vez del
# nombre YA CONCATENADO. Ahí está la diferencia que importa: el ETL arma
# `nombre_completo_derivada_rnpdno` con `fillna("")`, así que a quien no trae
# segundo apellido le queda un nombre de aspecto completo que el criterio daría
# por válido. Aquí un campo vacío es un dato
# faltante, que es lo que el recuadro dice contar.
#
# Son, entonces, dos condiciones:
#   1. le falta alguno de los tres campos (nombre, primer y segundo apellido), y
#   2. el criterio de nombre sobre la concatenación: sin dígitos, sin texto de
#      relleno y de 11 caracteres o más (el que el notebook 1 calculaba como
#      `es_nombre_valido`; ya no lo exporta porque sólo se usa aquí).
# Sólo con (2) el marcador daría 78; con las dos, ~1,708 dentro de `sisf`.
# El grupo va como `(?:...)` —no captura— sólo para que pandas no avise; el
# patrón es, alternativa por alternativa, el que usaba el notebook 1.
PATRON_NOMBRE_RUIDO = (
    r"SE DESCONOCE|RESTOS OSEOS|CUERPO|SIN VIDA|DESCONOCIDO|"
    r"NO IDENTIFICADO|\bSD\b|\bNA\b|^M$|(?:[a-z]\.){2,}[a-z]?"
)
COLS_NOMBRE_CRUDO = ["nombre_vic_rnpdno", "primerapellido_vic_rnpdno", "segundoapellido_vic_rnpdno"]


def col_cruda(d, candidatas):
    """Primera de `candidatas` presente en la rebanada. Falla fuerte si no hay."""
    for c in candidatas:
        if c in d.columns:
            return d[c]
    raise KeyError(
        f"Ninguna de {candidatas} está en el universo. Añádela a COLS_RN en "
        f"src/pipeline/derivadas.py y sube VERSION_DERIVADAS."
    )


def nombre_invalido(d):
    """Máscara de nombres que no pasan el criterio, sobre los campos crudos.

    `flags=re.IGNORECASE` no es intercambiable con `case=False`: con `case=False`
    pandas resuelve el patrón con el motor de Arrow, cuyo `\b` es ASCII y parte
    palabras en la `Ñ` —"NAÑEZ" dispararía el `\bNA\b` del patrón de ruido—.
    Con `flags` cae al `re` de Python, que es el que usaba el ETL del notebook 1.
    """
    partes = [d[c].fillna("").astype(str).str.strip() if c in d.columns
              else pd.Series("", index=d.index) for c in COLS_NOMBRE_CRUDO]
    falta_campo = partes[0].eq("") | partes[1].eq("") | partes[2].eq("")
    nom = (partes[0] + " " + partes[1] + " " + partes[2]) \
        .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
    return (falta_campo
            | nom.str.contains(r"\d", regex=True)
            | nom.str.contains(PATRON_NOMBRE_RUIDO, flags=re.IGNORECASE, regex=True)
            | (nom.str.len() < 11)).to_numpy()


CAMPOS_FALTANTES = {
    "smuni": lambda d: d["idmunicipio_derivada_rnpdno"].isna(),
    "sfnac": lambda d: d["fechanacimiento_derivada_rnpdno"].isna(),
    "sfdes": lambda d: col_cruda(d, COLS_FDES_CRUDA).isna(),
    "snomb": nombre_invalido,
    "ssexo": lambda d: d["sexo_rnpdno"].isna() | d["sexo_rnpdno"].astype(str).str.strip().eq(""),
}

# ── Comisiones nacional vs. local (lámina 3) ─────────────────────────────────
# `nnal`/`lcal` tampoco son banderas del pipeline: parten `is_com` por el nombre
# de la dependencia. La Comisión Nacional de Búsqueda es la única que dice
# "NACIONAL"; las 32 restantes son "COMISION LOCAL DE BUSQUEDA…".
COM_NACIONAL = "NACIONAL"

# ── Gráfica de la lámina 4 ───────────────────────────────────────────────────
# La plantilla ya trae los ejes, el título y la leyenda dibujados; lo que falta
# es el área de barras. Las tres series y sus colores salen de esa leyenda:
#   Fiscalía con carpeta de investigación  → cisf_sind_fis_cfval
#   Reporte de fiscalía                    → el resto de cisf_sind_fis
#   Localizados                            → loc_fis, la misma población que el
#                                            recuadro "Localizados de fiscalías"
# El orden del diccionario fija el apilado y, con él, el de `colores`.
SERIES_LAMINA4 = {
    "Fiscalía con carpeta de investigación": S_FIS.con(subfiltro="cisf_sind_fis_cfval"),
    "Reporte de fiscalía":                   S_FIS,        # se le resta cfval
    "Localizados":                           S_LOC_FIS_06,
}
SERIE_RESTA = ("Reporte de fiscalía", "Fiscalía con carpeta de investigación")

# Nombre de la gráfica; cada entidad le agrega su sufijo. Es también el marcador
# que se busca en la plantilla: `GR_ALIAS` lista los nombres que ha tenido el
# recuadro y la sección 5 usa el que efectivamente esté en el PPTX.
GR_LAMINA4 = "gr_anio_rn_01ene06-max_des_cisf_sind_fis"
GR_ALIAS = [GR_LAMINA4, "gr_mes_rn_01ene06-max_des_cisf_sind_fis"]

# Los parámetros de la gráfica no se declaran aquí: van escritos en la llamada a
# `barras_verticales` de la sección 6, que es donde se ajustan.

# Medidas de la especificación. `barras_verticales` recibe CENTÍMETROS y divide
# entre 2.54 internamente, así que el `/2.54` las convierte una segunda vez: la
# figura sale de 7.0 x 2.8 cm y, como el motor la inserta forzando el tamaño del
# recuadro, PowerPoint la estira 2.6x y las etiquetas se encabalgan. Con
# `AJUSTAR_A_RECUADRO = True` se dibuja a la medida real del cuadro de la
# plantilla —misma proporción, 2.48 contra 2.47— y el texto sale a su tamaño.
# Ponlo en False para usar la especificación al pie de la letra.
ANCHO_ESPEC_CM, ALTO_ESPEC_CM = 17.86 / 2.54, 7.2 / 2.54
AJUSTAR_A_RECUADRO = True

## 3. Universo

Acierto de caché si el notebook 4, el 6 o cualquier otra PPT con el mismo perfil de capa A
ya lo construyó y desde entonces no cambió `config.py`, el perfil, el código ni ningún
insumo; si cambió, se recalcula aquí. El universo es nacional y se rebana 33 veces; nunca se
reconstruye por entidad.

In [ ]:
u = Universo.construir(cfg)

## 4. Indicadores por entidad

Tres piezas:

- `acotar` convierte una rebanada nacional en la de la entidad.
- `extras_entidad` calcula lo que no es un conteo de rebanada: los datos faltantes, el
  reparto nacional/local de las comisiones, los conteos del último mes y del último año,
  los porcentajes `_v2` (cind y sind contra `cisf`, para que sumen 100 %) y el año del
  registro más antiguo de los localizados de fiscalías.
- `indicadores_entidad` junta todo y **quita el sufijo `_entNN`** de las claves, porque
  los marcadores de la plantilla llevan el nombre nacional.

El detalle que hay que cuidar son los porcentajes: `Slice.padre` quita primero la
entidad, así que dejar la regla del padre a su aire mediría la entidad contra el total
nacional. Por eso el denominador se acota SIEMPRE a la misma entidad que el numerador.

In [ ]:
# "Último mes": el último mes COMPLETO respecto al corte del RNPDNO. El mes en
# curso lleva unos pocos días de registros y como titular no dice nada; para
# usarlo igualmente, quita el `- 1`.
MES_UMES = pd.Timestamp(u.meta["fecha_corte_rnpdno"]).to_period("M") - 1
# "Último año", en cambio, es el año EN CURSO del corte, no el último completo:
# es el que rotula la lámina 4 ("… casos de fiscalías en {{fha_rn_anio}}").
ANIO_UANIO = pd.Timestamp(u.meta["fecha_corte_rnpdno"]).year
print(f"Último mes: {MES_UMES}  ·  último año: {ANIO_UANIO}  "
      f"(corte RNPDNO {u.meta['fecha_corte_rnpdno']})")


def acotar(s: Slice, cve):
    """La misma rebanada, acotada a una entidad. `None` = nacional."""
    return s if cve is None else s.por_entidad(cve)


def pct_texto(num: int, den: int, decimales: int = 1) -> str:
    """Mismo formato que `Universo.porcentaje`, para lo que se cuenta a mano."""
    return f"{100 * num / den:.{decimales}f}%" if den else "n/d"


def extras_entidad(cve) -> dict:
    """Marcadores que no son un conteo de rebanada, para una entidad."""
    ex = {"ent": config.ENTIDADES.get(cve, "Nacional") if cve else "Nacional"}

    # ── Datos faltantes dentro de `sisf` (lámina 2) ─────────────────────
    d = u.slice(acotar(S_SISF, cve))
    for token, prueba in CAMPOS_FALTANTES.items():
        ex[f"{S_SISF.nombre}_{token}"] = int(prueba(d).sum())

    # ── Comisiones: nacional vs. local (lámina 3) ───────────────────────
    d = u.slice(acotar(S_COM, cve))
    es_nnal = col_cruda(d, ["dependencia_origen_derivada_rnpdno_banci", "dependencia_origen_rnpdno"]) \
        .astype(str).str.upper().str.contains(COM_NACIONAL)
    for token, mascara in (("nnal", es_nnal), ("lcal", ~es_nnal)):
        n = int(mascara.sum())
        ex[f"{S_COM.nombre}_{token}"] = n
        ex[f"pct_{S_COM.nombre}_{token}"] = pct_texto(n, len(d))
    # La plantilla escribe este marcador con una `l` de más; se emite con las dos
    # grafías para que la lámina salga completa hoy y siga saliendo cuando se
    # corrija el PPTX (ver la sección 8).
    ex[f"pct_{S_COM.nombre}_nnall"] = ex[f"pct_{S_COM.nombre}_nnal"]

    # ── Último mes y último año (lámina 4) ──────────────────────────────
    # `_umes` y `_uanio` son la misma rebanada acotada al último mes completo y
    # al año en curso del corte. La plantilla usa las dos cosas en la misma
    # frase; ver la sección 8.
    for s in (S_FIS, S_FIS.con(subfiltro="cisf_sind_fis_cfval"),
              S_FIS.con(subfiltro="cisf_sind_fis_sfval")):
        fechas = u.slice(acotar(s, cve))[u.columna(s, "fecha")]
        ex[f"{s.nombre}_umes"] = int((fechas.dt.to_period("M") == MES_UMES).sum())
        ex[f"{s.nombre}_uanio"] = int((fechas.dt.year == ANIO_UANIO).sum())

    # ── Porcentajes `_v2`: cind y sind contra `cisf`, para que sumen 100 % ──
    den = u.contar(acotar(S_CISF, cve))
    for s in PORCENTAJES_V2:
        ex[f"pct_{s.nombre}_v2"] = pct_texto(u.contar(acotar(s, cve)), den)

    # ── Fechas que la plantilla nombra distinto ─────────────────────────
    # `Universo.indicadores` ya emite las variantes de fha_rn_*, fha_ba_* y
    # fha_hoy_* —incluida fha_hoy_dia_mes_anio—; aquí queda sólo la que no puede
    # emitir, porque no es una fecha de corte sino el mínimo de una rebanada.
    corte = u.meta["fecha_corte_rnpdno"]
    # `fha_rn_min-max_loc_fis_anio`: el año del registro más antiguo de esa
    # rebanada, que es lo que completa el paréntesis "(N desde ….)" de la lámina 4.
    anios = u.slice(acotar(S_LOC_FIS_MM, cve))[u.columna(S_LOC_FIS_MM, "fecha")].dt.year.dropna()
    ex[f"fha_rn_{S_LOC_FIS_MM.periodo}_loc_fis_anio"] = (
        str(int(anios.min())) if len(anios) else corte[:4])
    return ex


def indicadores_entidad(cve) -> dict:
    """Conteos + porcentajes + extras de una entidad, con las claves nacionales."""
    conteos = [acotar(s, cve) for s in CONTEOS]
    porcentajes = []
    for item in PORCENTAJES:
        # Acepta un Slice (denominador = su padre) o (Slice, denominador). El
        # denominador se resuelve en nacional y DESPUÉS se acota, porque
        # `Slice.padre` quitaría la entidad antes que el subfiltro.
        s, den = item if isinstance(item, tuple) else (item, item.padre)
        porcentajes.append((acotar(s, cve), acotar(den, cve)))

    vars_ppt = u.indicadores(conteos, porcentajes=porcentajes)
    if cve is not None:
        sufijo = f"_ent{cve:02d}"
        vars_ppt = {k.replace(sufijo, ""): v for k, v in vars_ppt.items()}
    vars_ppt.update(extras_entidad(cve))
    return vars_ppt

In [ ]:
# Vista previa: la nacional sirve de control (sus cifras son las del reporte diario).
vars_nacional = indicadores_entidad(None)
for k, v in vars_nacional.items():
    print(f"  {k:<50} {v}")

## 5. Auditoría de la plantilla

Antes de armar 33 archivos conviene saber si lo declarado y lo que pide el PPTX coinciden.
El cruce se lee en dos direcciones:

- **En la plantilla y sin calcular**: marcadores que saldrían literales en la lámina.
  Deben ser cero.
- **Calculadas y sin marcador**: claves de más. Algunas son inevitables —
  `Universo.indicadores` siempre emite las tres variantes de fecha de `rn` y `ba` —
  y otras señalan un marcador roto en la plantilla (sección 8).

In [ ]:
def marcadores_plantilla(ruta) -> set:
    """Todos los `{{marcador}}` del PPTX, incluidos los de formas agrupadas."""
    encontrados = set()

    def buscar(shapes):
        for shape in shapes:
            if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
                buscar(shape.shapes)
            elif shape.has_text_frame:
                # Por párrafo y no por run: un marcador partido en varios runs
                # (pasa al editar el texto) se reconstruye en `paragraph.text`.
                for parrafo in shape.text_frame.paragraphs:
                    encontrados.update(re.findall(r"{{(.*?)}}", parrafo.text))

    for lamina in Presentation(str(ruta)).slides:
        buscar(lamina.shapes)
    return encontrados


EN_PLANTILLA = marcadores_plantilla(PLANTILLA)
calculadas = set(vars_nacional)

# Marcador de imagen con el que se rotuló el recuadro de la lámina 4. Se toma el
# que esté en el PPTX: el nombre de la gráfica cambió a `gr_anio_…` y la plantilla
# puede seguir con el anterior.
MARCADOR_GRAFICA = next((m for m in GR_ALIAS if m in EN_PLANTILLA), GR_LAMINA4)
if MARCADOR_GRAFICA != GR_LAMINA4:
    print(f"ℹ️ El recuadro de la lámina 4 se llama {{{{{MARCADOR_GRAFICA}}}}} en la "
          f"plantilla; la gráfica se llama {GR_LAMINA4}. Conviene renombrarlo.\n")

print(f"{len(EN_PLANTILLA)} marcadores en {PLANTILLA.name} · {len(calculadas)} claves calculadas\n")
# El marcador de la gráfica es de IMAGEN: lo resuelve la sección 6, no `vars_ppt`.
sin_calcular = sorted(EN_PLANTILLA - calculadas - set(GR_ALIAS))
print(f"⚠️ En la plantilla y sin calcular ({len(sin_calcular)}): {sin_calcular or '—'}")
de_mas = sorted(calculadas - EN_PLANTILLA)
print(f"ℹ️ Calculadas y sin marcador ({len(de_mas)}): {de_mas}")

## 6. Gráfica de la lámina 4

Serie anual apilada de `cisf_sind_fis`, con las tres series de la leyenda que la plantilla
trae dibujada y el polígono de zoom apuntando al tramo reciente.

El recuadro de la lámina 4 lleva un marcador de imagen, así que la gráfica entra por la
vía normal del motor: sustituye al cuadro de texto que lo contiene, heredando su posición
y sus medidas, igual que en el notebook 6. La figura se genera con las medidas de ESE
cuadro (`dimensiones_placeholder`), porque el motor la inserta forzando ancho y alto: con
otra proporción, PowerPoint la deformaría.

Si el marcador se pierde al editar la plantilla, el cuaderno no se queda sin gráfica:
`insertar_grafica` la coloca por geometría, en el cuadro de texto vacío más grande de la
lámina.

Los parámetros van escritos en la llamada a `barras_verticales`, dentro de
`grafica_entidad`: es una sola gráfica repetida por entidad, así que ajustarla es editar
esa lista de argumentos.

In [ ]:
# `exportar_grafica` (1_funciones/funciones.py) tiene "output/<nombre>" codificado y
# relativo al directorio de trabajo, así que los PNG y SVG caen en notebooks/output/
# sin importar lo que se declare aquí. RUTA_GRAFICAS apunta a esa ruta real para que
# el armado de la PPT encuentre las imágenes.
RUTA_GRAFICAS = Path("output")
RUTA_GRAFICAS.mkdir(parents=True, exist_ok=True)

LAMINA_GRAFICA = 4          # 1-based, como se numeran las láminas en el PPTX
ANCHO_MIN_CAJA_CM = 10.0    # descarta los recuadros pequeños que también están vacíos


def caja_grafica(ruta, lamina: int = LAMINA_GRAFICA):
    """La forma vacía donde va la gráfica: el cuadro de texto sin texto más grande.

    Se filtra por TEXT_BOX a propósito: las cápsulas decorativas de la lámina son
    autoformas y también tienen el texto vacío.
    """
    diapo = Presentation(str(ruta)).slides[lamina - 1]
    cajas = [sh for sh in diapo.shapes
             if sh.shape_type == MSO_SHAPE_TYPE.TEXT_BOX
             and not sh.text_frame.text.strip()
             and Emu(sh.width).cm >= ANCHO_MIN_CAJA_CM]
    return max(cajas, key=lambda s: s.width * s.height) if cajas else None


def tabla_anual(cve) -> pd.DataFrame:
    """Serie por año de desaparición con las tres series de la leyenda.

    Los años sin registros se rellenan con cero: sin eso, un hueco desaparece del
    eje y la serie aparenta continuidad donde no la hay.
    """
    columnas = {etiqueta: u.slice(acotar(s, cve))[u.columna(s, "fecha")].dt.year.value_counts()
                for etiqueta, s in SERIES_LAMINA4.items()}
    t = pd.DataFrame(columnas).fillna(0).astype(int)
    if t.empty:
        return t.rename_axis("anio").reset_index()
    # "Reporte de fiscalía" es el total de fiscalías menos las que traen carpeta:
    # apiladas, las dos series reconstruyen `cisf_sind_fis`.
    total, parte = SERIE_RESTA
    t[total] = (t[total] - t[parte]).clip(lower=0)
    t = t.reindex(range(int(t.index.min()), int(t.index.max()) + 1), fill_value=0)
    return t.rename_axis("anio").reset_index()


def grafica_entidad(cve, ancho_cm: float, alto_cm: float) -> Path:
    """Dibuja la gráfica de una entidad con las medidas dadas y la guarda.

    Los parámetros van escritos aquí, en la llamada, y no en un diccionario de
    estilo aparte: es UNA gráfica —la de la lámina 4— repetida por entidad, así
    que ajustarla es editar esta lista de argumentos. Los tres de
    `poligono_zoom` se portaron a 1_funciones/barras_verticales.py desde la
    versión anterior de la librería.
    """
    nombre = f"{GR_LAMINA4}_{'nacional' if cve is None else f'ent{cve:02d}'}"
    barras_verticales(
        tabla_anual(cve), nombre=nombre, mostrar=False,
        ancho=ancho_cm, alto=alto_cm,
        colores=["#af5967", "#4f3230", "#b5c9c4"],
        tam_valor_ejex=6.0,
        tam_valor_ejey=8,
        tam_valor_barra=7,
        porce_barra=False,
        valor_capsu=False,
        ordenar="x",              # el eje X son años: `ordenar="x"` los pone cronológicos
        areamin=100,
        ajustar_sep_valor_barra=2.5,
        leyenda=False,            # la leyenda ya está dibujada en la plantilla
        tam_letra_leyenda=7,
        poligono_zoom=True,
        posiciones_poligono_zoom=[
            (4, 0.95, 0.23),      # Vértice 4: 95% ancho de la figura, 23% de la altura
            (5, 0.95, 0.55),      # Vértice 5: 95% ancho de la figura, 55% de la altura
        ],
        opacidad_poligono_zoom=0.8,
    )
    return RUTA_GRAFICAS / f"{nombre}.png"


def insertar_grafica(ruta_pptx, imagen, lamina: int = LAMINA_GRAFICA) -> bool:
    """Sustituye el recuadro vacío de la lámina por la imagen. `False` si no lo hay."""
    prs = Presentation(str(ruta_pptx))
    diapo = prs.slides[lamina - 1]
    cajas = [sh for sh in diapo.shapes
             if sh.shape_type == MSO_SHAPE_TYPE.TEXT_BOX
             and not sh.text_frame.text.strip()
             and Emu(sh.width).cm >= ANCHO_MIN_CAJA_CM]
    if not cajas:
        return False
    caja = max(cajas, key=lambda s: s.width * s.height)
    x, y, cx, cy = caja.left, caja.top, caja.width, caja.height
    caja._element.getparent().remove(caja._element)
    diapo.shapes.add_picture(str(imagen), x, y, cx, cy)
    prs.save(str(ruta_pptx))
    return True


# Medidas con las que se dibuja la figura: las del cuadro que lleva el marcador y,
# si alguien lo borró, las del recuadro vacío que quede en su lugar.
if MARCADOR_GRAFICA in EN_PLANTILLA:
    ANCHO_CAJA_CM, ALTO_CAJA_CM = dimensiones_placeholder(PLANTILLA, MARCADOR_GRAFICA)
else:
    _caja = caja_grafica(PLANTILLA)
    ANCHO_CAJA_CM, ALTO_CAJA_CM = ((Emu(_caja.width).cm, Emu(_caja.height).cm)
                                   if _caja is not None else (None, None))
    if _caja is not None:
        print(f"ℹ️ La plantilla no trae el marcador {{{{{MARCADOR_GRAFICA}}}}}; la gráfica se "
              f"colocará por geometría en el recuadro vacío de la lámina {LAMINA_GRAFICA}.")

if ANCHO_CAJA_CM is None:
    ANCHO_GRAFICA_CM = ALTO_GRAFICA_CM = None
    print(f"⚠️ En la lámina {LAMINA_GRAFICA} de {PLANTILLA.name} no hay ni marcador ni "
          f"recuadro vacío para la gráfica; no se generará.")
else:
    ANCHO_GRAFICA_CM, ALTO_GRAFICA_CM = ((ANCHO_CAJA_CM, ALTO_CAJA_CM)
                                         if AJUSTAR_A_RECUADRO
                                         else (ANCHO_ESPEC_CM, ALTO_ESPEC_CM))
    print(f"📐 Cuadro de la lámina {LAMINA_GRAFICA}: {ANCHO_CAJA_CM:.2f} x "
          f"{ALTO_CAJA_CM:.2f} cm · figura {ANCHO_GRAFICA_CM:.2f} x {ALTO_GRAFICA_CM:.2f} cm")
    # Vista previa con la nacional. `mostrar=False` evita que matplotlib pinte su
    # figura en vivo: lo que se muestra es el PNG ya guardado, que es exactamente
    # el archivo que se inserta en la plantilla.
    display(Image(filename=str(grafica_entidad(None, ANCHO_GRAFICA_CM, ALTO_GRAFICA_CM))))

## 7. Armado de las presentaciones

Una PPTX por entidad, todas desde la misma plantilla y el mismo universo. Al final se
imprime el resumen: marcadores sustituidos, marcadores que la plantilla no tenía y si la
gráfica encontró su recuadro.

In [ ]:
def nombre_archivo(cve) -> str:
    """`Reporte_Estatal_17_Morelos.pptx`; la nacional va como 00 para que ordene primero."""
    if cve is None:
        return "Reporte_Estatal_00_Nacional.pptx"
    texto = unicodedata.normalize("NFD", config.ENTIDADES[cve])
    texto = texto.encode("ascii", "ignore").decode("ascii")
    return f"Reporte_Estatal_{cve:02d}_{'_'.join(texto.split())}.pptx"


SALIDA_DIR.mkdir(parents=True, exist_ok=True)
generadas, incidencias = [], {}

for cve in ENTIDADES_A_GENERAR:
    etiqueta = config.ENTIDADES.get(cve, "Nacional") if cve else "Nacional"
    vars_ppt = indicadores_entidad(cve)
    salida = SALIDA_DIR / nombre_archivo(cve)

    builder = PPTReportBuilder(PLANTILLA)
    # Las claves van sin llaves: en la plantilla el marcador es {{reg_rn_min-max}}.
    builder.add_text_variables(vars_ppt)
    imagen = (grafica_entidad(cve, ANCHO_GRAFICA_CM, ALTO_GRAFICA_CM)
              if ANCHO_GRAFICA_CM else None)
    if imagen and MARCADOR_GRAFICA in EN_PLANTILLA:
        builder.add_image_variables({MARCADOR_GRAFICA: imagen})
    reporte = builder.render(salida, verbose=False)

    # Respaldo: sólo si la plantilla perdió el marcador y quedó el recuadro vacío.
    grafica = bool(reporte["image_report"].get(MARCADOR_GRAFICA))
    if imagen and not grafica:
        grafica = insertar_grafica(salida, imagen)

    faltantes = sorted(k for k, n in reporte["text_report"].items()
                       if n == 0 and k in EN_PLANTILLA and k not in GR_ALIAS)
    if faltantes:
        incidencias[etiqueta] = faltantes
    generadas.append(salida)
    print(f"  ✅ {etiqueta:<24} {salida.name:<42} "
          f"{sum(1 for n in reporte['text_report'].values() if n)} marcadores"
          f"{' + gráfica' if grafica else ' · SIN GRÁFICA'}")

print(f"\n📦 {len(generadas)} presentaciones en {SALIDA_DIR}")
if incidencias:
    print(f"⚠️ Marcadores de la plantilla sin sustituir: {incidencias}")